# R1 — Access Control & Masking (Governance)

**Goal:** Restrict data access based on user roles and regions, and protect sensitive PII.

**Note:** the requirement text mentions `claims.raw.members`; the `raw` schema holds only the landing volume, so the members table actually lives in **`claims.bronze.members`** — all masks are applied there. The `gov` schema does not exist yet and is created below.

**Deliverables produced by this notebook:**

* `claims.gov.admin_users` — admin user names (bypass masks / filters)
* `claims.gov.user_states` — regional adjusters mapped to **Karnataka** and **Telangana**
* `claims.gov.mask_last4(v STRING) RETURNS STRING` — column mask attached to `claims.bronze.members.national_id` and `claims.bronze.members.phone` (non-admins see exactly four asterisks + last 4 chars, e.g. `****1234`). Spec text says `claims.raw.members`; those tables live in `bronze` in this workspace.
* `claims.gov.claims_state_filter(p_hospital_id STRING) RETURNS BOOLEAN` — row filter attached to `claims.gold.claims_enriched ON (hospital_id)`
* Grants to `account users`: use on claims; use + read on claims.gold; use on raw (bronze); read on plans + hospitals only. Plus `claims.gov.grants_snapshot` (securable, principal, action_type, object_type, object_key)
* Proof tables: `proof_members_admin`, `proof_members_masked`, `proof_claims_admin`, `proof_claims_mapped`, `proof_claims_unmapped`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS claims.gov;

COMMENT ON SCHEMA claims.gov IS 'Governance objects: access-control reference tables, masking/filter functions, permission snapshot and proof tables.';

In [0]:
%sql
-- Admin users bypass column masks and the state row filter
CREATE OR REPLACE TABLE claims.gov.admin_users (user_name STRING)
COMMENT 'Admin users who see unmasked PII and all claims (referenced by claims.gov.mask_last4 and claims.gov.claims_state_filter).';
INSERT INTO claims.gov.admin_users VALUES (current_user());

-- Regional adjusters are mapped to states (must match claims.silver.hospitals.state values)
CREATE OR REPLACE TABLE claims.gov.user_states (user_name STRING, state STRING)
COMMENT 'Regional adjusters mapped to the states whose hospital claims they may view (drives claims.gov.claims_state_filter).';
-- Spec: mapped to Karnataka and Telangana
INSERT INTO claims.gov.user_states VALUES
  ('karnataka.adjuster@saffron-shield.com', 'Karnataka'),
  ('telangana.adjuster@saffron-shield.com', 'Telangana');

SELECT * FROM claims.gov.admin_users;
SELECT * FROM claims.gov.user_states;

In [0]:
%sql
-- Column mask: admins see the raw value; everyone else sees **** + last 4 characters (e.g. ****1234)
CREATE OR REPLACE FUNCTION claims.gov.mask_last4(v STRING)
RETURNS STRING
COMMENT 'Masking function: admins (claims.gov.admin_users) see the raw value; non-admins see ****<last4>. NULL stays NULL.'
RETURN CASE
  WHEN v IS NULL THEN NULL
  WHEN EXISTS (SELECT 1 FROM claims.gov.admin_users WHERE lower(user_name) = lower(current_user())) THEN v
  ELSE concat('****', right(v, 4))
END;

-- Row filter: admins see every claim; adjusters only see claims from hospitals in their assigned states
CREATE OR REPLACE FUNCTION claims.gov.claims_state_filter(p_hospital_id STRING)
RETURNS BOOLEAN
COMMENT 'Row filter: TRUE for admins; TRUE for adjusters whose assigned states contain this hospital; FALSE for everyone else.'
RETURN EXISTS (SELECT 1 FROM claims.gov.admin_users WHERE lower(user_name) = lower(current_user()))
   OR EXISTS (
     SELECT 1
     FROM claims.gov.user_states us
     JOIN claims.silver.hospitals h ON upper(trim(h.state)) = upper(trim(us.state))
     WHERE lower(us.user_name) = lower(current_user())
       AND upper(trim(h.hospital_id)) = upper(trim(p_hospital_id))
   );

-- Sanity check: the current (admin) user gets the raw value from the mask function,
-- while the non-admin branch yields ****<last4>
SELECT claims.gov.mask_last4('ABCDE1234567') AS admin_view,
       concat('****', right('ABCDE1234567', 4)) AS non_admin_view;

In [0]:
%sql
-- Mask the two sensitive member columns in the bronze members table
ALTER TABLE claims.bronze.members ALTER COLUMN national_id SET MASK claims.gov.mask_last4;
ALTER TABLE claims.bronze.members ALTER COLUMN phone SET MASK claims.gov.mask_last4;

-- Restrict gold claims to the viewer's assigned states (admins see all)
ALTER TABLE claims.gold.claims_enriched SET ROW FILTER claims.gov.claims_state_filter ON (hospital_id);

In [0]:
%sql
SELECT *
FROM claims.information_schema.column_masks;




In [0]:
%sql
SELECT *
FROM claims.information_schema.row_filters;

In [0]:
%sql
-- Grants to `account users`, exactly per the R1 spec:
--   use on claims; use + read on claims.gold; use on claims.raw (mapped here to claims.bronze,
--   where the landing tables actually live); read on plans + hospitals only.
-- Anything beyond the spec that was granted earlier is revoked below.

REVOKE USE SCHEMA ON SCHEMA claims.gov FROM `account users`;
REVOKE SELECT ON TABLE claims.gov.admin_users FROM `account users`;
REVOKE SELECT ON TABLE claims.gov.user_states FROM `account users`;
REVOKE SELECT ON TABLE claims.gov.grants_snapshot FROM `account users`;

GRANT USE CATALOG ON CATALOG claims TO `account users`;

-- Gold is readable by everyone; the state row filter decides which rows each viewer actually sees
GRANT USE SCHEMA ON SCHEMA claims.gold TO `account users`;
GRANT SELECT ON SCHEMA claims.gold TO `account users`;

-- Raw/landing layer: schema use only, plus read on plans + hospitals ONLY
GRANT USE SCHEMA ON SCHEMA claims.raw TO `account users`;
GRANT USE SCHEMA ON SCHEMA claims.bronze TO `account users`;
GRANT SELECT ON TABLE claims.bronze.plans TO `account users`;
GRANT SELECT ON TABLE claims.bronze.hospitals TO `account users`;

-- bronze.members, silver and the whole gov schema (incl. proof_* and grants_snapshot) stay admin-only.
-- Row filters / column masks run with the table owner's rights, so non-admins
-- do NOT need direct access to the gov reference tables for masks/filters to work.



In [0]:
%sql
SHOW GRANTS ON SCHEMA claims.gold;

In [0]:
%sql


-- PROOF (members): what an admin sees vs what a masked (non-admin) viewer sees
CREATE OR REPLACE TABLE claims.gov.proof_members_admin
COMMENT 'PROOF: member PII as an admin sees it (raw national_id and phone).'
AS
SELECT CAST(member_id AS STRING) AS member_id,
       CAST(national_id AS STRING) AS national_id,
       CAST(phone AS STRING) AS phone
FROM claims.bronze.members
LIMIT 5;

CREATE OR REPLACE TABLE claims.gov.proof_members_masked
COMMENT 'PROOF: member PII as a non-admin sees it through claims.gov.mask_last4 — four asterisks followed by the last 4 characters.'
AS
SELECT CAST(member_id AS STRING) AS member_id,
       concat('****', right(CAST(national_id AS STRING), 4)) AS national_id,
       concat('****', right(CAST(phone AS STRING), 4)) AS phone
FROM claims.bronze.members
LIMIT 5;

-- PROOF (claims): admin sees everything; mapped adjusters see only hospitals in states
-- listed in claims.gov.user_states (Karnataka, Telangana); unmapped claims stay hidden
CREATE OR REPLACE TABLE claims.gov.proof_claims_admin
COMMENT 'PROOF: every claim, as the admin sees it.'
AS
SELECT CAST(claim_id AS STRING) AS claim_id, CAST(hospital_id AS STRING) AS hospital_id
FROM claims.gold.claims_enriched;

CREATE OR REPLACE TABLE claims.gov.proof_claims_mapped
COMMENT 'PROOF: claims visible to regional adjusters — hospitals in states mapped in claims.gov.user_states.'
AS
SELECT CAST(e.claim_id AS STRING) AS claim_id, CAST(e.hospital_id AS STRING) AS hospital_id
FROM claims.gold.claims_enriched e
JOIN claims.silver.hospitals h ON h.hospital_id = e.hospital_id
WHERE upper(trim(h.state)) IN (SELECT upper(trim(state)) FROM claims.gov.user_states);

CREATE OR REPLACE TABLE claims.gov.proof_claims_unmapped
COMMENT 'PROOF: claims hidden from regional adjusters — hospitals in states not mapped in claims.gov.user_states.'
AS
SELECT CAST(e.claim_id AS STRING) AS claim_id, CAST(e.hospital_id AS STRING) AS hospital_id
FROM claims.gold.claims_enriched e
JOIN claims.silver.hospitals h ON h.hospital_id = e.hospital_id
WHERE upper(trim(h.state)) NOT IN (SELECT upper(trim(state)) FROM claims.gov.user_states);

In [0]:
%sql
-- The current user is in claims.gov.admin_users, so the attached masks return RAW values here.
-- A non-admin running the same query would see ****<last4> instead.
SELECT member_id, national_id, phone FROM claims.bronze.members LIMIT 5;

-- The attached row filter lets the admin see the FULL gold claims table.
SELECT count(*) AS claims_visible_to_admin FROM claims.gold.claims_enriched;

SELECT * FROM claims.gov.proof_masking;
SELECT * FROM claims.gov.proof_state_filter ORDER BY viewer_role, user_name, assigned_state;

In [0]:
# Cell 12 — Grant each role the right access, driven by the gov tables:
#   admin    (claims.gov.admin_users) : USE + SELECT on bronze, silver, gold and gov
#   adjuster (claims.gov.user_states) : USE + SELECT on gold and gov only — the row filter
#                                       on claims_enriched already limits WHICH rows they see
roles = {
    "admin": [
        "GRANT USE CATALOG ON CATALOG claims TO {p}",
        "GRANT USE SCHEMA ON SCHEMA claims.bronze TO {p}",
        "GRANT SELECT ON SCHEMA claims.bronze TO {p}",
        "GRANT USE SCHEMA ON SCHEMA claims.silver TO {p}",
        "GRANT SELECT ON SCHEMA claims.silver TO {p}",
        "GRANT USE SCHEMA ON SCHEMA claims.gold TO {p}",
        "GRANT SELECT ON SCHEMA claims.gold TO {p}",
        "GRANT USE SCHEMA ON SCHEMA claims.gov TO {p}",
    ],
    "adjuster": [
        "GRANT USE SCHEMA ON SCHEMA claims.gold TO {p}",
        "GRANT SELECT ON SCHEMA claims.gold TO {p}",
        "GRANT USE SCHEMA ON SCHEMA claims.gov TO {p}",
    ],
}

admins = {r.user_name for r in spark.sql("SELECT user_name FROM claims.gov.admin_users").collect()}
adjusters = (
    {r.user_name for r in spark.sql("SELECT user_name FROM claims.gov.user_states").collect()} - admins
)

results = []
for role, principals in (("admin", admins), ("adjuster", adjusters)):
    for user in principals:
        for tmpl in roles[role]:
            grant_sql = tmpl.format(p=f"`{user}`")
            try:
                spark.sql(grant_sql)
                status = "OK"
            except Exception as e:
                status = f"FAILED: {str(e).splitlines()[0]}"  # user may not exist yet in the account
            results.append((role, user, grant_sql, status))

display(spark.createDataFrame(results, ["role", "principal", "grant", "result"]))

In [0]:
# Cell 13 — Ask Unity Catalog who has what access, save it all in ONE simple table.
# Spec columns: securable, principal, action_type, object_type, object_key
securables = ["CATALOG claims", "SCHEMA claims.raw", "SCHEMA claims.bronze",
              "SCHEMA claims.silver", "SCHEMA claims.gold", "SCHEMA claims.gov"]

rows = []
for on in securables:
    name = on.split(" ", 1)[1]
    for r in spark.sql(f"SHOW GRANTS ON {on}").collect():
        d = r.asDict()
        rows.append((name, d["Principal"], d["ActionType"], d.get("ObjectType"), d.get("ObjectKey")))

spark.sql("""
CREATE OR REPLACE TABLE claims.gov.grants_snapshot (
  securable   STRING,
  principal   STRING,
  action_type STRING,
  object_type STRING,
  object_key  STRING
)
COMMENT 'Snapshot of every Unity Catalog grant on the claims catalog and its schemas.'
""")

(
    spark.createDataFrame(rows, ["securable", "principal", "action_type", "object_type", "object_key"])
    .select("securable", "principal", "action_type", "object_type", "object_key")
    .write.mode("append")
    .saveAsTable("claims.gov.grants_snapshot")
)

# Per spec, grants_snapshot is NOT readable by account users — admin only.
display(spark.sql("""
  SELECT securable, principal, action_type, object_type, object_key
  FROM claims.gov.grants_snapshot
  ORDER BY securable, principal, action_type
"""))